# Example 16.3 / 1: Generate a 4 × 4 Puzzle

Companion notebook for **16.3 Sudoku Generator** in the AMPL lab Documentation, section "Generate a 4 × 4 Puzzle". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Generate a 4 × 4 Puzzle

Keep the Sudoku model in AMPL and the clue-removal logic in Python. The run starts from the complete grid and visits the 16 cells in row-major order. For each cell it sets the clue to 0 with `ampl.param["Given"]`, solves with HiGHS, and reads `ampl.solve_result`. It keeps the removal only when the status is `infeasible`; any other status, including `solved` or a time limit, restores the clue. Each record row stores the cell, old clue, solve status and decision:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param Base integer > 0 default 3;  # box side b; the grid is Base^2 x Base^2
param GridSize := Base^2;  # digits per row, column and box
set ROWS := 1..GridSize;  # grid rows
set COLS := 1..GridSize;  # grid columns
set DIGITS := 1..GridSize;  # digits that can fill a cell
set BOX {br in 1..Base, bc in 1..Base} =  # cells of the box in box row br, box column bc
    {(br-1)*Base+1..br*Base, (bc-1)*Base+1..bc*Base};

param Template {ROWS, COLS} integer >= 1, <= GridSize;  # known completion to exclude
param Given {ROWS, COLS} integer >= 0, <= GridSize default 0;  # clue digit; 0 is an empty cell
var x {ROWS, COLS, DIGITS} binary;  # 1 if cell (r,c) holds digit d

subject to OneDigit {r in ROWS, c in COLS}:  # exactly one digit per cell
    sum {d in DIGITS} x[r,c,d] = 1;
subject to RowOnce {r in ROWS, d in DIGITS}:  # each digit once per row
    sum {c in COLS} x[r,c,d] = 1;
subject to ColumnOnce {c in COLS, d in DIGITS}:  # each digit once per column
    sum {r in ROWS} x[r,c,d] = 1;
subject to BoxOnce {br in 1..Base, bc in 1..Base, d in DIGITS}:  # each digit once per box
    sum {(r,c) in BOX[br,bc]} x[r,c,d] = 1;
subject to KeepGiven {r in ROWS, c in COLS: Given[r,c] > 0}:  # clues stay fixed
    x[r,c,Given[r,c]] = 1;
subject to DifferentFromTemplate:  # no-good cut: differ from Template somewhere
    sum {r in ROWS, c in COLS} x[r,c,Template[r,c]] <= card(ROWS) * card(COLS) - 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["Base"] = 2
completion = [[2, 1, 4, 3], [4, 3, 2, 1], [1, 2, 3, 4], [3, 4, 1, 2]]
template = {(r, c): completion[r - 1][c - 1] for r in range(1, 5) for c in range(1, 5)}
ampl.param["Template"] = template

In [5]:
# run.py
given = dict(template)
record = []
for cell in sorted(template):
    given[cell] = 0
    ampl.param["Given"] = given
    ampl.solve(solver="highs", verbose=False)
    accepted = ampl.solve_result == "infeasible"
    if not accepted:
        given[cell] = template[cell]
    record.append((cell, template[cell], ampl.solve_result, accepted))

for cell, old, status, accepted in record:
    print(cell, old, status, "removed" if accepted else "kept")
print("clues:", {k: v for k, v in given.items() if v > 0})

(1, 1) 2 infeasible removed
(1, 2) 1 infeasible removed
(1, 3) 4 infeasible removed
(1, 4) 3 infeasible removed
(2, 1) 4 infeasible removed
(2, 2) 3 infeasible removed
(2, 3) 2 solved kept
(2, 4) 1 solved kept
(3, 1) 1 infeasible removed
(3, 2) 2 solved kept
(3, 3) 3 infeasible removed
(3, 4) 4 solved kept
(4, 1) 3 infeasible removed
(4, 2) 4 solved kept
(4, 3) 1 solved kept
(4, 4) 2 infeasible removed
clues: {(2, 3): 2, (2, 4): 1, (3, 2): 2, (3, 4): 4, (4, 2): 4, (4, 3): 1}


### Pure AMPL: data.dat and model.run

The same loop in pure AMPL, with the same `model.mod`. `option presolve 0` passes every infeasible model to HiGHS; with the default presolve, AMPL prints presolve warnings for each infeasible case and, in a test run, stopped the loop after `Bailing out after 10 warnings`. The `suffix dunbdd OUT;` line declares the suffix in which HiGHS returns information about an infeasible model; without it, AMPL announces the new suffix in the log:

In [6]:
%%writefile data.dat
param Base := 2;
param Template:  1 2 3 4 :=
             1   2 1 4 3
             2   4 3 2 1
             3   1 2 3 4
             4   3 4 1 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0, presolve 0;
suffix dunbdd OUT;
let {r in ROWS, c in COLS} Given[r,c] := Template[r,c];
for {r in ROWS, c in COLS} {
    let Given[r,c] := 0;
    solve;
    if solve_result <> "infeasible" then let Given[r,c] := Template[r,c];
    printf "(%d,%d) %s\n", r, c, solve_result;
}
display Given;

(1,1) infeasible
(1,2) infeasible
(1,3) infeasible
(1,4) infeasible
(2,1) infeasible
(2,2) infeasible
(2,3) solved
(2,4) solved
(3,1) infeasible
(3,2) solved
(3,3) infeasible
(3,4) solved
(4,1) infeasible
(4,2) solved
(4,3) solved
(4,4) infeasible
Given :=
1 1   0
1 2   0
1 3   0
1 4   0
2 1   0
2 2   0
2 3   2
2 4   1
3 1   0
3 2   2
3 3   0
3 4   4
4 1   0
4 2   4
4 3   1
4 4   0
;



### Another way in amplpy

The first visit removes clue `(1,1)` from the full grid and the cut model is `infeasible`: with 15 clues, the template is the only completion. Every kept clue has a `solved` status, so the alternative model found a second completion when that clue was missing.

> **A time limit is not a uniqueness proof**
> If the alternative search stops without a proof of infeasibility, conservatively restore the clue. "No second solution found yet" and "no second solution exists" are different statements.

The visiting order changes the result. The run below, with the same model.mod and data.py, wraps the loop in a function and compares three orders:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["Base"] = 2
completion = [[2, 1, 4, 3], [4, 3, 2, 1], [1, 2, 3, 4], [3, 4, 1, 2]]
template = {(r, c): completion[r - 1][c - 1] for r in range(1, 5) for c in range(1, 5)}
ampl.param["Template"] = template

In [11]:
# run.py
def generate(order):
    given = dict(template)
    for cell in order:
        given[cell] = 0
        ampl.param["Given"] = given
        ampl.solve(solver="highs", verbose=False)
        if ampl.solve_result != "infeasible":
            given[cell] = template[cell]
    return sorted(k for k, v in given.items() if v > 0)


cells = sorted(template)
orders = {
    "row-major": cells,
    "reverse row-major": cells[::-1],
    "column-major": sorted(cells, key=lambda rc: (rc[1], rc[0])),
}
for label, order in orders.items():
    print(f"{label}: {generate(order)}")

row-major: [(2, 3), (2, 4), (3, 2), (3, 4), (4, 2), (4, 3)]
reverse row-major: [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
column-major: [(2, 3), (2, 4), (3, 2), (3, 4), (4, 2), (4, 3)]
